In [0]:
%run "/Users/scottsbv@gmail.com/student-engagement/Dea Student Engagement Pipeline/silver_engine_utils"

In [0]:
# ============================================================================
# SILVER CHILD TABLE — lead_activites_raw_body_text_quoted
# ============================================================================
# Purpose: Quoted text body segments from email activities
# This table is NEEDED when:
#   - You need to analyze quoted reply text in email threads
#   - You're tracking email conversation context
#   - You're building email threading features
# ============================================================================

from pyspark.sql.functions import col, explode_outer

parent_table = "crm_ingestion.silver.lead_activites_raw"
child_table  = "crm_ingestion.silver.lead_activites_raw_body_text_quoted"

print("\n" + "="*80)
print("🔥 BUILDING CHILD TABLE — lead_activites_raw_body_text_quoted")
print("="*80)

df_parent = spark.table(parent_table)

# Explode the body_text_quoted array of structs into individual rows
df_child = (
    df_parent
        .filter(col("body_text_quoted").isNotNull())
        .withColumn("_element", explode_outer(col("body_text_quoted")))
        .select(
            col("id").alias("lead_activites_raw_id"),
            col("_element.expand"),
            col("_element.text"),
            col("bronze_insert_date"),
            col("bronze_updated_at")
        )
)

df_child.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(child_table)

print(f"\n✅ Table created: {child_table}")
print(f"🔢 Row count: {df_child.count():,}")

print("\n📘 SCHEMA:")
df_child.printSchema()

print("\n📊 SAMPLE ROWS (5):")
display(spark.table(child_table).limit(5))